# Audio FIM LM: 1D RoPE Demo

This notebook is a small sandbox for trying the positional idea from `audio_fim_lm_outline.py`.

The important experiment here is that **input/cache order** and **temporal position** do not have to be the same:

- cache/input order can be: `history, prefix, suffix, middle-query`
- temporal positions can still be: `history..., prefix, middle..., suffix`

That is the key trick that lets a causal/KV-cache model see the right anchor early while still treating it as the future boundary.

In [1]:
import math
import torch
import torch.nn as nn

torch.manual_seed(7)
torch.set_printoptions(precision=4, sci_mode=False)

## 1. Minimal 1D RoPE

This is not WAN's 3D RoPE. For motion tokens, we only need one temporal axis.

RoPE is applied to `q` and `k`, not added to token embeddings. The shape convention below is:

```text
q, k: (batch, seq_len, num_heads, head_dim)
position_ids: (batch, seq_len)
```

In [2]:
class RotaryEmbedding1D(nn.Module):
    def __init__(self, head_dim, base=10000.0):
        super().__init__()
        assert head_dim % 2 == 0, "RoPE head_dim must be even"
        inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, position_ids):
        # position_ids: (B, T)
        freqs = torch.einsum("bt,d->btd", position_ids.float(), self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        cos = emb.cos().unsqueeze(2)  # (B, T, 1, head_dim)
        sin = emb.sin().unsqueeze(2)  # (B, T, 1, head_dim)
        return cos, sin


def rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(q, k, cos, sin):
    q = (q * cos) + (rotate_half(q) * sin)
    k = (k * cos) + (rotate_half(k) * sin)
    return q, k

## 2. Fake FIM Layout

Suppose we already generated `H=3` history frames, and the current gap has `N=4` missing frames.

Temporal timeline:

```text
history0, history1, history2, prefix, middle0, middle1, middle2, middle3, suffix
pos 0     pos 1     pos 2     pos 3   pos 4    pos 5    pos 6    pos 7    pos 8
```

But for FIM/KV cache, input order may be:

```text
history0, history1, history2, prefix, suffix, middle_query0
```

Notice that suffix appears early in cache order, but keeps temporal position `8`.

In [3]:
H = 3
N = 4

tokens = [
    "history0",
    "history1",
    "history2",
    "prefix_left",
    "suffix_right",      # appears early in input/cache order
    "middle_query0",     # first target frame query
]

temporal_position_ids = torch.tensor([[
    0,          # history0
    1,          # history1
    2,          # history2
    H,          # prefix_left
    H + N + 1,  # suffix_right, future boundary
    H + 1,      # middle_query0
]])

cache_order_position_ids = torch.arange(len(tokens)).unsqueeze(0)

for i, name in enumerate(tokens):
    print(f"{i:02d} cache_order={i:02d} temporal_pos={temporal_position_ids[0, i].item():02d} token={name}")

00 cache_order=00 temporal_pos=00 token=history0
01 cache_order=01 temporal_pos=01 token=history1
02 cache_order=02 temporal_pos=02 token=history2
03 cache_order=03 temporal_pos=03 token=prefix_left
04 cache_order=04 temporal_pos=08 token=suffix_right
05 cache_order=05 temporal_pos=04 token=middle_query0


## 3. Apply RoPE to Fake Q/K

Here we create fake hidden states and project them to Q/K. The exact values do not matter; this cell is just to check shapes and see how the temporal `position_ids` are used.

In [4]:
B = 1
T = len(tokens)
hidden_size = 32
num_heads = 4
head_dim = hidden_size // num_heads

hidden_states = torch.randn(B, T, hidden_size)
q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
k_proj = nn.Linear(hidden_size, hidden_size, bias=False)

q = q_proj(hidden_states).view(B, T, num_heads, head_dim)
k = k_proj(hidden_states).view(B, T, num_heads, head_dim)

rope = RotaryEmbedding1D(head_dim=head_dim)
cos, sin = rope(temporal_position_ids)
q_rot, k_rot = apply_rope(q, k, cos, sin)

print("q shape:", tuple(q.shape))
print("cos shape:", tuple(cos.shape))
print("q_rot shape:", tuple(q_rot.shape))

q shape: (1, 6, 4, 8)
cos shape: (1, 6, 1, 8)
q_rot shape: (1, 6, 4, 8)


## 4. Compare Temporal Positions vs Cache-Order Positions

This is the main sanity check. If you accidentally use `torch.arange(seq_len)` as the RoPE position, suffix will get position `4`, even though temporally it should be position `8`.

For FIM, the suffix should keep the future temporal position.

In [5]:
cos_temporal, sin_temporal = rope(temporal_position_ids)
q_temporal, k_temporal = apply_rope(q, k, cos_temporal, sin_temporal)

cos_cache_order, sin_cache_order = rope(cache_order_position_ids)
q_cache_order, k_cache_order = apply_rope(q, k, cos_cache_order, sin_cache_order)

suffix_idx = tokens.index("suffix_right")
diff = (k_temporal[:, suffix_idx] - k_cache_order[:, suffix_idx]).abs().mean()

print("suffix cache-order position:", cache_order_position_ids[0, suffix_idx].item())
print("suffix temporal position:   ", temporal_position_ids[0, suffix_idx].item())
print("mean abs difference in rotated suffix key:", float(diff))

suffix cache-order position: 4
suffix temporal position:    8
mean abs difference in rotated suffix key: 0.2744576930999756


C:\Users\WenJChai\AppData\Local\Temp\ipykernel_57516\3700751269.py:12: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:836.)
  print("mean abs difference in rotated suffix key:", float(diff))


## 5. Tiny Cache-Style Attention Demo

This mimics the FIM/KV-cache idea:

1. Prefill known context: history + prefix + suffix.
2. Query the next middle frame.
3. The query attends to cached context.

This is not the full model. It is just the attention math shape.

In [6]:
known_count = 5  # history0, history1, history2, prefix_left, suffix_right
query_idx = 5

# Convert to attention layout: (B, heads, T, head_dim)
k_cache = k_temporal[:, :known_count].transpose(1, 2)
v_cache = torch.randn(B, known_count, num_heads, head_dim).transpose(1, 2)
q_query = q_temporal[:, query_idx : query_idx + 1].transpose(1, 2)

scores = torch.matmul(q_query, k_cache.transpose(-2, -1)) / math.sqrt(head_dim)
weights = torch.softmax(scores, dim=-1)
context = torch.matmul(weights, v_cache)

print("k_cache:", tuple(k_cache.shape))
print("q_query:", tuple(q_query.shape))
print("attention weights:", tuple(weights.shape))
print("context:", tuple(context.shape))

print("\nAttention over cached tokens, averaged over heads:")
avg_weights = weights.mean(dim=1)[0, 0]
for name, w in zip(tokens[:known_count], avg_weights):
    print(f"{name:>14s}: {float(w):.4f}")

k_cache: (1, 4, 5, 8)
q_query: (1, 4, 1, 8)
attention weights: (1, 4, 1, 5)
context: (1, 4, 1, 8)

Attention over cached tokens, averaged over heads:
      history0: 0.2048
      history1: 0.1648
      history2: 0.2062
   prefix_left: 0.2004
  suffix_right: 0.2238


## Takeaway

For the future `AudioFIMLM`:

- use RoPE position IDs to represent **temporal motion position**;
- do not blindly use cache/input order as position;
- suffix/right anchor can be cached early, but should keep its future temporal position;
- segment embeddings can separately tell the model whether a token is history, prefix, suffix, audio, or middle query.